# OpenSees Cantilever Pushover — PyLauncher Sweep

This notebook runs a parameter sweep over a 2D cantilever pushover analysis using
PyLauncher and dapi. We sweep over `NodalMass` and `LCol` (column length) to study
how these structural parameters affect the pushover response.

The cantilever model:
```
   ^Y
   |
   2       __
   |         |
   |         |
   |         |
 (1)      LCol
   |         |
   |         |
   |         |
 =1=    ----  -------->X
```

- Node 1: fixed base
- Node 2: free top with `NodalMass`
- Elastic beam-column element
- Gravity load (2000 kip downward) followed by lateral pushover (displacement-controlled)

For a simpler introduction to PyLauncher sweeps, see [pylauncher_sweep.ipynb](pylauncher_sweep.ipynb).

In [ ]:
%pip install dapi --quiet

In [ ]:
import os
from pathlib import Path
from dapi import DSClient

ds = DSClient()

MYDATA = Path(os.environ.get("JUPYTER_SERVER_ROOT", os.path.expanduser("~"))) / "MyData"

## Write the analysis script

An OpenSeesPy cantilever pushover script based on the
[OpenSees Examples Manual](https://opensees.berkeley.edu/wiki/index.php/Examples_Manual).
It accepts `--NodalMass`, `--LCol`, and `--outDir` as command-line arguments
so PyLauncher can run each parameter combination independently.

In [ ]:
input_dir = MYDATA / "opensees_sweep"
input_dir.mkdir(parents=True, exist_ok=True)

cantilever_script = """\
# Ex1a.Canti2D.Push — OpenSeesPy cantilever pushover
# Based on the OpenSees Examples Manual
# Units: kip, inch, second
#
# Command-line arguments (set by PyLauncher per task):
#   --NodalMass  mass at free node
#   --LCol       column length
#   --outDir     output directory for this run

import argparse
import os

if os.path.exists("opensees.so"):
    import opensees as ops
else:
    import openseespy.opensees as ops

parser = argparse.ArgumentParser()
parser.add_argument("--NodalMass", type=float, required=True)
parser.add_argument("--LCol", type=float, required=True)
parser.add_argument("--outDir", type=str, required=True)
args = parser.parse_args()

NodalMass = args.NodalMass
LCol = args.LCol
outDir = args.outDir

os.makedirs(outDir, exist_ok=True)
print(f"Running: NodalMass={NodalMass}, LCol={LCol}, outDir={outDir}")

ops.wipe()
ops.model("basic", "-ndm", 2, "-ndf", 3)

# Geometry
ops.node(1, 0, 0)
ops.node(2, 0, LCol)
ops.fix(1, 1, 1, 1)
ops.mass(2, NodalMass, 0.0, 0.0)

# Element
ops.geomTransf("Linear", 1)
ops.element("elasticBeamColumn", 1, 1, 2, 3600000000, 4227, 1080000, 1)

# Recorders
ops.recorder("Node", "-file", f"{outDir}/DFree.out", "-time", "-node", 2, "-dof", 1, 2, 3, "disp")
ops.recorder("Node", "-file", f"{outDir}/RBase.out", "-time", "-node", 1, "-dof", 1, 2, 3, "reaction")
ops.recorder("Element", "-file", f"{outDir}/FCol.out", "-time", "-ele", 1, "globalForce")

# Gravity analysis
ops.timeSeries("Linear", 1)
ops.pattern("Plain", 1, 1)
ops.load(2, 0.0, -2000.0, 0.0)
ops.wipeAnalysis()
ops.constraints("Plain")
ops.numberer("Plain")
ops.system("BandGeneral")
ops.test("NormDispIncr", 1.0e-8, 6)
ops.algorithm("Newton")
ops.integrator("LoadControl", 0.1)
ops.analysis("Static")
ops.analyze(10)
ops.loadConst("-time", 0.0)

# Pushover analysis
ops.timeSeries("Linear", 2)
ops.pattern("Plain", 2, 2)
ops.load(2, 2000.0, 0.0, 0.0)
ops.integrator("DisplacementControl", 2, 1, 0.1)
ops.analyze(1000)

print(f"Done: NodalMass={NodalMass}, LCol={LCol}")
"""

(input_dir / "cantilever.py").write_text(cantilever_script)
print(f"Wrote {input_dir}/cantilever.py")

## Define the sweep

5 nodal masses x 3 column lengths = 15 independent analyses.

In [ ]:
sweep = {
    "NODAL_MASS": [4.19, 4.39, 4.59, 4.79, 4.99],
    "LCOL": [100, 200, 300],
}

## Preview

In [ ]:
df = ds.jobs.parametric_sweep.generate(
    "python3 cantilever.py --NodalMass NODAL_MASS --LCol LCOL --outDir out_NODAL_MASS_LCOL",
    sweep,
    preview=True,
)
print(f"Total runs: {len(df)}")
df

## Generate sweep files

In [ ]:
commands = ds.jobs.parametric_sweep.generate(
    "python3 cantilever.py --NodalMass NODAL_MASS --LCol LCOL --outDir out_NODAL_MASS_LCOL",
    sweep,
    str(input_dir),
)

print(f"Generated {len(commands)} task commands\n")
print("=== runsList.txt ===")
print((input_dir / "runsList.txt").read_text())

print("=== call_pylauncher.py ===")
print((input_dir / "call_pylauncher.py").read_text())

print("=== Files in input directory ===")
for fn in sorted(os.listdir(input_dir)):
    print(f"  {fn}")

## Submit

Replace `your_allocation` with your TACC allocation and uncomment to run.

In [ ]:
# job = ds.jobs.parametric_sweep.submit(
#     "/MyData/opensees_sweep/",
#     app_id="designsafe-agnostic-app",
#     allocation="your_allocation",
#     node_count=1,
#     cores_per_node=48,
#     max_minutes=30,
# )
# job.monitor()